In [ ]:
import kagglehub
import shutil
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Get project root
BASE_DIR = Path().resolve().parent

data_dir = BASE_DIR / "data"
outputs_dir = BASE_DIR / "outputs"

# Create directories if they do not exist
data_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

In [ ]:
# Define dataset file
file_name = "winemag-data-130k-v2.csv"
target_file = data_dir / file_name

# Download only if dataset is not already in data folder
if not target_file.exists():
    path = kagglehub.dataset_download("zynicide/wine-reviews")
    print("Dataset downloaded in user/.cache/kagglehub/datasets/zynicide/")

    source_file = Path(path) / file_name
    shutil.copy(source_file, target_file)
    print("Dataset copied to data/")
else:
    print("Dataset already exists in data folder")

# Load data
df_wine = pd.read_csv(target_file)
df_wine.head()

In [ ]:
# Extract vintage year from title using regex
target = "title"
year_pattern = r"(19\d{2}|20\d{2})"

df_wine["year"] = df_wine[target].str.extract(year_pattern, expand=False)

df_wine["year"] = pd.to_numeric(
    df_wine["year"],
    errors="coerce"
)

print(df_wine[["title", "year"]].head(10))

In [ ]:
# Remove records before 2004
mask_year = df_wine["year"] >= 2004
df_wine = df_wine[mask_year].copy()

In [ ]:
# Calculate missing values percentage
print("Missing values:")
null_data = (df_wine.isnull().sum() / len(df_wine)) * 100
print(null_data[null_data > 0].sort_values(ascending=False))

In [ ]:
# Select relevant columns
cols = [
    "country",
    "description",
    "designation",
    "points",
    "price",
    "province",
    "region_1",
    "title",
    "variety",
    "year"
]

df_wine = df_wine[cols]
df_wine.head()

In [ ]:
# Drop records with missing values in key columns
required_cols = [
    "year",
    "variety",
    "price",
    "designation"
]

df_wine = df_wine.dropna(subset=required_cols).copy()
df_wine = df_wine.reset_index(drop=True)

In [ ]:
# Calculate missing values percentage after cleaning
print("Missing values:")
null_data = (df_wine.isnull().sum() / len(df_wine)) * 100
print(null_data[null_data > 0].sort_values(ascending=False))

In [ ]:
# Check number of wines by country
country_counts = df_wine["country"].value_counts().head(15).reset_index()
country_counts.columns = ["country", "count"]

plt.figure(figsize=(12, 8))

sns.barplot(
    data=country_counts,
    x="count",
    y="country"
)

In [ ]:
# Randomly sample wines to limit the maximum dataset size
countries_list = df_wine["country"].value_counts().head(10).index.tolist()

# Estimate maximum records per country
max_records = 10000 // len(countries_list)

sampled_dfs = []

for country in countries_list:
    df_temp = df_wine[df_wine["country"] == country]

    if len(df_temp) > max_records:
        sampled_dfs.append(df_temp.sample(n=max_records, random_state=42))
    else:
        sampled_dfs.append(df_temp)

df_wine = pd.concat(sampled_dfs).reset_index(drop=True)

In [ ]:
# Check number of wines by country after sampling
country_counts = df_wine["country"].value_counts().head(15).reset_index()
country_counts.columns = ["country", "count"]

plt.figure(figsize=(12, 8))

sns.barplot(
    data=country_counts,
    x="count",
    y="country"
)

In [ ]:
# Translate country names into English for dashboard visualization
df_wine["country"] = df_wine["country"].replace(
    {
        "US":                "United States",
        "Italy":             "Italy",
        "France":            "France",
        "Spain":             "Spain",
        "Germany":           "Germany",
        "South Africa":      "South Africa",
        "New Zealand":       "New Zealand",
        "Greece":            "Greece",
    }
)

In [ ]:
# Classify wine quality based on points
def classify_quality(points):
    if points >= 98:
        return "6/6 Outstanding"
    if points >= 94:
        return "5/6 Superior"
    if points >= 90:
        return "4/6 Excellent"
    if points >= 87:
        return "3/6 Very Good"
    if points >= 83:
        return "2/6 Good"
    return "1/6 Acceptable"


df_wine["category"] = df_wine["points"].apply(classify_quality)

In [ ]:
# Graph 1: wine count by country and quality category
df_graph1 = df_wine.groupby(["country", "category"]).size().reset_index(name="count")

df_graph1.columns = ["Country", "Quality", "Count"]
df_graph1.to_csv(outputs_dir / "graph_1.csv", index=False)

In [ ]:
# Graph 2: average price by country and quality category
df_graph2 = df_wine.groupby(["country", "category"])["price"].mean().reset_index()
df_graph2["price"] = df_graph2["price"].round(2)

df_graph2.columns = ["Country", "Quality", "Average Price ($)"]
df_graph2.to_csv(outputs_dir / "graph_2.csv", index=False)

In [ ]:
# Graph 3: top wines by price range
bins = [0, 10, 20, 50, 100, 500, 1500]
labels = ["Up to 10€", "Up to 20€", "Up to 50€", "Up to 100€", "Up to 500€", "Up to 1500€"]

df_wine["price_range"] = pd.cut(df_wine["price"], bins=bins, labels=labels)

df_top = df_wine.sort_values(by=["points", "price"], ascending=[False, True])
df_top = df_top.groupby("price_range", observed=True).head(3)

df_graph3 = pd.DataFrame()
df_graph3["Name"] = df_top["variety"]
df_graph3["Image"] = ""
df_graph3["Country"] = df_top["country"]
df_graph3["Price Range"] = df_top["price_range"]
df_graph3["Score"] = df_top["points"].astype(str) + " pts"
df_graph3["Price"] = df_top["price"].astype(str) + "€"

df_graph3.to_csv(outputs_dir / "graph_3.csv", index=False)

In [ ]:
# Graph 4: unique wine varieties by country for map visualization
official_country_mapping = {
    "US":                              "United States of America",
    "England":                         "United Kingdom",
    "Macedonia":                       "Republic of Macedonia",
    "Germany":                         "Germany",
    "Italy":                           "Italy",
    "France":                          "France",
    "Spain":                           "Spain",
    "Portugal":                        "Portugal",
    "Australia":                       "Australia",
    "Austria":                         "Austria",
    "Argentina":                       "Argentina",
    "Chile":                           "Chile",
    "South Africa":                    "South Africa",
    "New Zealand":                     "New Zealand",
    "Israel":                          "Israel",
    "Hungary":                         "Hungary",
    "Greece":                          "Greece",
    "Romania":                         "Romania",
    "Mexico":                          "Mexico",
    "Canada":                          "Canada",
    "Turkey":                          "Turkey",
    "Czech Republic":                  "Czech Republic",
    "Slovenia":                        "Slovenia",
    "Luxembourg":                      "Luxembourg",
    "Croatia":                         "Croatia",
    "Georgia":                         "Georgia",
    "Uruguay":                         "Uruguay",
    "Lebanon":                         "Lebanon",
    "Serbia":                          "Serbia",
    "Brazil":                          "Brazil",
    "Moldova":                         "Moldova",
    "Morocco":                         "Morocco",
    "Peru":                            "Peru",
    "India":                           "India",
    "Bulgaria":                        "Bulgaria",
    "Cyprus":                          "Cyprus",
    "Armenia":                         "Armenia",
    "Switzerland":                     "Switzerland",
    "Bosnia and Herzegovina":          "Bosnia and Herzegovina",
    "Ukraine":                         "Ukraine",
    "Slovakia":                        "Slovakia",
    "China":                           "China",
    "Egypt":                           "Egypt"
}

df_map = df_wine.dropna(subset=["country"]).copy()
df_map["Country name"] = df_map["country"].map(official_country_mapping).fillna(df_map["country"])

df_graph4 = df_map.groupby("Country name")["variety"].nunique().reset_index()
df_graph4.columns = ["Country name", "Variety count"]

df_graph4.to_csv(outputs_dir / "graph_4.csv", index=False)

In [ ]:
# Export cleaned dataset
clean_file = outputs_dir / "wine_cleaned.csv"
df_wine.to_csv(clean_file, index=False)

print(f"Clean dataset exported to outputs/")